<a href="https://colab.research.google.com/github/Bing-CRV/-unsloth-/blob/main/unsloth_transform_guff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2
import torch; torch._dynamo.config.recompile_limit = 64;


In [ ]:
from google.colab import runtime
runtime.unassign()


In [ ]:
# %%capture
# !pip install --no-deps --upgrade timm # Only for Gemma 3N

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,

)

In [ ]:

guff_path = "gemma-3-finetune"
lora_path = "/content/drive/MyDrive/models/gemma3f_lora"

# 使用 FastModel 提供的方法加载 LoRA
model = FastModel.get_peft_model(
    model,
    lora_path=lora_path,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

model.save_pretrained_merged(guff_path, tokenizer)
model.save_pretrained_gguf(
      guff_path,
      quantization_type = "Q8_0", # For now only Q8_0, BF16, F16 supported
      tokenizer = tokenizer
)
print(f"GGUF 保存完成：{guff_path}")


In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "gemma-3N-finetune",
        quantization_type = "Q8_0", # Only Q8_0, BF16, F16 supported
        repo_id = "HF_ACCOUNT/gemma-3N-finetune-gguf",
        token = "hf_...",
    )

In [ ]:
from google.colab import files
files.download("/content/gemma-3-finetune.Q8_0.gguf")

In [ ]:
import shutil

# 压缩成 ZIP，生成 models.zip
shutil.make_archive('model_g', 'zip','/content')


In [ ]:
shutil.copy('model_g.zip', '/content/drive/MyDrive/')